In [1]:
# Import Libraries and Modules
import os
import numpy as np
import pandas as pd

1.1 - Data Acquisition

In [2]:
# Base directory containing all original datasets
datasets_path = "data/original/" 

# CIC-UNSW-NB15 dataset directory
cicunswnb15 = os.path.join(datasets_path, "cic-unsw-nb15")

# CIC-IDS-2017 dataset directory
cicids2017 = os.path.join(datasets_path, "cic-ids-2017")

# CSE-CIC-IDS-2018 dataset directory
csecicids2018 = os.path.join(datasets_path, "cse-cic-ids-2018")

# CIC-DDoS-2019 dataset directory
cicddos2019 = (os.path.join(datasets_path, "cic-ddos-2019/03-11")), (os.path.join(datasets_path, "cic-ddos-2019/01-12"))

# Combined list of all dataset directories
datasets = [cicunswnb15, cicids2017, csecicids2018, cicddos2019[0], cicddos2019[1]]

In [ ]:
# Create dataset directories
for dataset in datasets:
    os.makedirs(dataset, exist_ok=True)

In [ ]:
# Store the total label count across all datasets
TotalCount = pd.Series(dtype="int")

# Iterate through each dataset directory
for dataset in datasets:
    print(f"Reading dataset {dataset}:")

    # Iterate through every CSV file in the dataset directory
    for file in os.listdir(dataset):

        # Store label count for the current file
        FileCount = pd.Series(dtype="int") 
        
        file_path = os.path.join(dataset, file)

        # Read CSV file in chunks to reduce memory usage
        for chunk in pd.read_csv(file_path, 
                                 usecols=["Label"], 
                                 chunksize=500_000, 
                                 skipinitialspace=True, 
                                 low_memory=False, 
                                 encoding="latin-1"):
            
            # Count occurrences of each label in the chunk
            valueCount = chunk["Label"].value_counts()
            
            # Add chunk counts to current file and overall dataset total
            FileCount = FileCount.add(valueCount, fill_value=0)
            TotalCount = TotalCount.add(valueCount, fill_value=0)
        
        # Print label distribution for current file 
        print(f"Value Count for {file}:")
        print(FileCount.astype(int).sort_values(ascending=False).to_string() + "\n")

# Print label distribution across all datasets
print("-" * 40)
print("\nTotal Label Counts:")
print(TotalCount.astype(int).sort_values(ascending=False).to_string())


Reading dataset data/original/cic-unsw-nb15:
Value Count for CICFlowMeter_out.csv:
Label
Benign            3450658
Exploits            30951
Fuzzers             29613
Reconnaissance      16735
Generic              4632
DoS                  4467
Shellcode            2102
Backdoor              452
Analysis              385
Worms                 246

Reading dataset data/original/cic-ids-2017:
Value Count for Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv:
Label
BENIGN          288566
Infiltration        36

Value Count for Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv:
Label
DDoS      128027
BENIGN     97718

Value Count for Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv:
Label
PortScan    158930
BENIGN      127537

Value Count for Tuesday-WorkingHours.pcap_ISCX.csv:
Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897

Value Count for Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv:
Label
BENIGN                           168186
Web Attack 

1.2 - Feature Standardisation

In [ ]:
# Original dataset columns

print("CIC-UNSW-NB15 Dataset:")
cicunswnb15_columns = pd.read_csv(f"{cicunswnb15}CICFlowMeter_out.csv", nrows=0)
print(cicunswnb15_columns)

print("\nCIC-IDS-2017 Dataset:")
cicids2017_columns = pd.read_csv(f"{cicids2017}Monday-WorkingHours.pcap_ISCX.csv", nrows=0)
print(cicids2017_columns)

print("\nCSE-CIC-IDS-2018 Dataset:")
csecicids2018_columns = pd.read_csv(f"{csecicids2018}02-14-2018.csv", nrows=0)
print(csecicids2018_columns)

print("\nCIC-DDOS-2019 Dataset:")
cicddos2019_columns = pd.read_csv(f"{cicddos2019[0]}Portmap.csv", nrows=0)
print(cicddos2019_columns)


CIC-UNSW-NB15 Dataset:
Empty DataFrame
Columns: [Flow ID, Src IP, Src Port, Dst IP, Dst Port, Protocol, Timestamp, Flow Duration, Total Fwd Packet, Total Bwd packets, Total Length of Fwd Packet, Total Length of Bwd Packet, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std, Bwd Packet Length Max, Bwd Packet Length Min, Bwd Packet Length Mean, Bwd Packet Length Std, Flow Bytes/s, Flow Packets/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Total, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Bwd IAT Total, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, Fwd Header Length, Bwd Header Length, Fwd Packets/s, Bwd Packets/s, Packet Length Min, Packet Length Max, Packet Length Mean, Packet Length Std, Packet Length Variance, FIN Flag Count, SYN Flag Count, RST Flag Count, PSH Flag Count, ACK Flag Count, URG Flag Count, CWR Flag Count, ECE Flag Count, Down/Up Rat

In [ ]:
def ColumnStandardisation(dataset, column_map):
    """ Standardises column names in all CSV files within the dataset directory """
    
    # Loops through all files in the dataset folder
    for file in os.listdir(dataset):

        # Only process CSV files
        if file.endswith(".csv"):

            file_path = os.path.join(dataset, file)
            tmp_file_path = file_path + ".tmp"

            # Track header to ensure no duplicate headers are written
            header_written = False 

            # Read CSV file in chunks
            for chunk in pd.read_csv(file_path, low_memory=False, chunksize=500_000, skipinitialspace=True, encoding="latin-1"):
                
                # Rename columns based on mapping
                chunk.rename(columns=column_map, inplace=True)
                
                # Write chunk to temporary file
                chunk.to_csv(tmp_file_path, mode='a', index=False, header=not header_written)
                
                # After first chunk, set header as written
                header_written = True

            # Replace original file with processed file
            os.replace(tmp_file_path, file_path)
            print(f"Finished processing {file}")

In [ ]:
cicunswnb15_column_map = {
    "Flow ID": "Flow ID",
    "Src IP": "Src IP",
    "Src Port": "Src Port",
    "Dst IP": "Dst IP",
    "Dst Port": "Dst Port",
    "Protocol": "Protocol",
    "Timestamp": "Timestamp",
    "Flow Duration": "Flow Duration",
    "Total Fwd Packet": "Total Fwd Packet",
    "Total Bwd packets": "Total Bwd packets",
    "Total Length of Fwd Packet": "Total Length of Fwd Packet",
    "Total Length of Bwd Packet": "Total Length of Bwd Packet",
    "Fwd Packet Length Max": "Fwd Packet Length Max",
    "Fwd Packet Length Min": "Fwd Packet Length Min",
    "Fwd Packet Length Mean": "Fwd Packet Length Mean",
    "Fwd Packet Length Std": "Fwd Packet Length Std",
    "Bwd Packet Length Max": "Bwd Packet Length Max",
    "Bwd Packet Length Min": "Bwd Packet Length Min",
    "Bwd Packet Length Mean": "Bwd Packet Length Mean",
    "Bwd Packet Length Std": "Bwd Packet Length Std",
    "Flow Bytes/s": "Flow Bytes/s",
    "Flow Packets/s": "Flow Packets/s",
    "Flow IAT Mean": "Flow IAT Mean",
    "Flow IAT Std": "Flow IAT Std",
    "Flow IAT Max": "Flow IAT Max",
    "Flow IAT Min": "Flow IAT Min",
    "Fwd IAT Total": "Fwd IAT Total",
    "Fwd IAT Mean": "Fwd IAT Mean",
    "Fwd IAT Std": "Fwd IAT Std",
    "Fwd IAT Max": "Fwd IAT Max",
    "Fwd IAT Min": "Fwd IAT Min",
    "Bwd IAT Total": "Bwd IAT Total",
    "Bwd IAT Mean": "Bwd IAT Mean",
    "Bwd IAT Std": "Bwd IAT Std",
    "Bwd IAT Max": "Bwd IAT Max",
    "Bwd IAT Min": "Bwd IAT Min",
    "Fwd PSH Flags": "Fwd PSH Flags",
    "Bwd PSH Flags": "Bwd PSH Flags",
    "Fwd URG Flags": "Fwd URG Flags",
    "Bwd URG Flags": "Bwd URG Flags",
    "Fwd Header Length": "Fwd Header Length",
    "Bwd Header Length": "Bwd Header Length",
    "Fwd Packets/s": "Fwd Packets/s",
    "Bwd Packets/s": "Bwd Packets/s",
    "Packet Length Min": "Packet Length Min",
    "Packet Length Max": "Packet Length Max",
    "Packet Length Mean": "Packet Length Mean",
    "Packet Length Std": "Packet Length Std",
    "Packet Length Variance": "Packet Length Variance",
    "FIN Flag Count": "FIN Flag Count",
    "SYN Flag Count": "SYN Flag Count",
    "RST Flag Count": "RST Flag Count",
    "PSH Flag Count": "PSH Flag Count",
    "ACK Flag Count": "ACK Flag Count",
    "URG Flag Count": "URG Flag Count",
    "CWR Flag Count": "CWR Flag Count",
    "ECE Flag Count": "ECE Flag Count",
    "Down/Up Ratio": "Down/Up Ratio",
    "Average Packet Size": "Average Packet Size",
    "Fwd Segment Size Avg": "Fwd Segment Size Avg",
    "Bwd Segment Size Avg": "Bwd Segment Size Avg",
    "Fwd Bytes/Bulk Avg": "Fwd Bytes/Bulk Avg",
    "Fwd Packet/Bulk Avg": "Fwd Packet/Bulk Avg",
    "Fwd Bulk Rate Avg": "Fwd Bulk Rate Avg",
    "Bwd Bytes/Bulk Avg": "Bwd Bytes/Bulk Avg",
    "Bwd Packet/Bulk Avg": "Bwd Packet/Bulk Avg",
    "Bwd Bulk Rate Avg": "Bwd Bulk Rate Avg",
    "Subflow Fwd Packets": "Subflow Fwd Packets",
    "Subflow Fwd Bytes": "Subflow Fwd Bytes",
    "Subflow Bwd Packets": "Subflow Bwd Packets",
    "Subflow Bwd Bytes": "Subflow Bwd Bytes",
    "FWD Init Win Bytes": "FWD Init Win Bytes",
    "Bwd Init Win Bytes": "Bwd Init Win Bytes",
    "Fwd Act Data Pkts": "Fwd Act Data Pkts",
    "Fwd Seg Size Min": "Fwd Seg Size Min",
    "Active Mean": "Active Mean",
    "Active Std": "Active Std",
    "Active Max": "Active Max",
    "Active Min": "Active Min",
    "Idle Mean": "Idle Mean",
    "Idle Std": "Idle Std",
    "Idle Max": "Idle Max",
    "Idle Min": "Idle Min",
    "Label": "Label"
}

# Call the ColumnStandardisation function for the CIC-UNSW-NB15 dataset
ColumnStandardisation(cicunswnb15, cicunswnb15_column_map)

cicunswnb15_columns = pd.read_csv(f"{cicunswnb15}/CICFlowMeter_out.csv", nrows=0)
print("\n" + cicunswnb15_columns)

Finished processing CICFlowMeter_out.csv
Empty DataFrame
Columns: [Flow ID, Src IP, Src Port, Dst IP, Dst Port, Protocol, Timestamp, Flow Duration, Total Fwd Packet, Total Bwd packets, Total Length of Fwd Packet, Total Length of Bwd Packet, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std, Bwd Packet Length Max, Bwd Packet Length Min, Bwd Packet Length Mean, Bwd Packet Length Std, Flow Bytes/s, Flow Packets/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Total, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Bwd IAT Total, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, Fwd Header Length, Bwd Header Length, Fwd Packets/s, Bwd Packets/s, Packet Length Min, Packet Length Max, Packet Length Mean, Packet Length Std, Packet Length Variance, FIN Flag Count, SYN Flag Count, RST Flag Count, PSH Flag Count, ACK Flag Count, URG Flag Count, CWR Flag Count, ECE Flag 

In [ ]:
cicids2017_column_map = {
    "Flow ID": "Flow ID",
    "Source IP": "Src IP",
    "Source Port": "Src Port",
    "Destination IP": "Dst IP",
    "Destination Port": "Dst Port",
    "Protocol": "Protocol",
    "Timestamp": "Timestamp",
    "Flow Duration": "Flow Duration",
    "Total Fwd Packets": "Total Fwd Packet",
    "Total Backward Packets": "Total Bwd packets",
    "Total Length of Fwd Packets": "Total Length of Fwd Packet",
    "Total Length of Bwd Packets": "Total Length of Bwd Packet",
    "Fwd Packet Length Max": "Fwd Packet Length Max",
    "Fwd Packet Length Min": "Fwd Packet Length Min",
    "Fwd Packet Length Mean": "Fwd Packet Length Mean",
    "Fwd Packet Length Std": "Fwd Packet Length Std",
    "Bwd Packet Length Max": "Bwd Packet Length Max",
    "Bwd Packet Length Min": "Bwd Packet Length Min",
    "Bwd Packet Length Mean": "Bwd Packet Length Mean",
    "Bwd Packet Length Std": "Bwd Packet Length Std",
    "Flow Bytes/s": "Flow Bytes/s",
    "Flow Packets/s": "Flow Packets/s",
    "Flow IAT Mean": "Flow IAT Mean",
    "Flow IAT Std": "Flow IAT Std",
    "Flow IAT Max": "Flow IAT Max",
    "Flow IAT Min": "Flow IAT Min",
    "Fwd IAT Total": "Fwd IAT Total",
    "Fwd IAT Mean": "Fwd IAT Mean",
    "Fwd IAT Std": "Fwd IAT Std",
    "Fwd IAT Max": "Fwd IAT Max",
    "Fwd IAT Min": "Fwd IAT Min",
    "Bwd IAT Total": "Bwd IAT Total",
    "Bwd IAT Mean": "Bwd IAT Mean",
    "Bwd IAT Std": "Bwd IAT Std",
    "Bwd IAT Max": "Bwd IAT Max",
    "Bwd IAT Min": "Bwd IAT Min",
    "Fwd PSH Flags": "Fwd PSH Flags",
    "Bwd PSH Flags": "Bwd PSH Flags",
    "Fwd URG Flags": "Fwd URG Flags",
    "Bwd URG Flags": "Bwd URG Flags",
    "Fwd Header Length": "Fwd Header Length",
    "Bwd Header Length": "Bwd Header Length",
    "Fwd Packets/s": "Fwd Packets/s",
    "Bwd Packets/s": "Bwd Packets/s",
    "Min Packet Length": "Packet Length Min",
    "Max Packet Length": "Packet Length Max",
    "Packet Length Mean": "Packet Length Mean",
    "Packet Length Std": "Packet Length Std",
    "Packet Length Variance": "Packet Length Variance",
    "FIN Flag Count": "FIN Flag Count",
    "SYN Flag Count": "SYN Flag Count",
    "RST Flag Count": "RST Flag Count",
    "PSH Flag Count": "PSH Flag Count",
    "ACK Flag Count": "ACK Flag Count",
    "URG Flag Count": "URG Flag Count",
    "CWE Flag Count": "CWR Flag Count",   
    "ECE Flag Count": "ECE Flag Count",
    "Down/Up Ratio": "Down/Up Ratio",
    "Average Packet Size": "Average Packet Size",
    "Avg Fwd Segment Size": "Fwd Segment Size Avg",
    "Avg Bwd Segment Size": "Bwd Segment Size Avg",
    "Fwd Avg Bytes/Bulk": "Fwd Bytes/Bulk Avg",
    "Fwd Avg Packets/Bulk": "Fwd Packet/Bulk Avg",
    "Fwd Avg Bulk Rate": "Fwd Bulk Rate Avg",
    "Bwd Avg Bytes/Bulk": "Bwd Bytes/Bulk Avg",
    "Bwd Avg Packets/Bulk": "Bwd Packet/Bulk Avg",
    "Bwd Avg Bulk Rate": "Bwd Bulk Rate Avg",
    "Subflow Fwd Packets": "Subflow Fwd Packets",
    "Subflow Fwd Bytes": "Subflow Fwd Bytes",
    "Subflow Bwd Packets": "Subflow Bwd Packets",
    "Subflow Bwd Bytes": "Subflow Bwd Bytes",
    "Init_Win_bytes_forward": "FWD Init Win Bytes",
    "Init_Win_bytes_backward": "Bwd Init Win Bytes",
    "act_data_pkt_fwd": "Fwd Act Data Pkts",
    "min_seg_size_forward": "Fwd Seg Size Min",
    "Active Mean": "Active Mean",
    "Active Std": "Active Std",
    "Active Max": "Active Max",
    "Active Min": "Active Min",
    "Idle Mean": "Idle Mean",
    "Idle Std": "Idle Std",
    "Idle Max": "Idle Max",
    "Idle Min": "Idle Min",
    "Label": "Label"
}

# Call the ColumnStandardisation function for the CIC-IDS-2017 dataset
ColumnStandardisation(cicids2017, cicids2017_column_map)

cicids2017_columns = pd.read_csv(f"{cicids2017}/Monday-WorkingHours.pcap_ISCX.csv", nrows=0)
print("\n" + cicids2017_columns)

Finished processing Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Finished processing Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Finished processing Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Finished processing Tuesday-WorkingHours.pcap_ISCX.csv
Finished processing Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Finished processing Wednesday-workingHours.pcap_ISCX.csv
Finished processing Monday-WorkingHours.pcap_ISCX.csv
Finished processing Friday-WorkingHours-Morning.pcap_ISCX.csv
Empty DataFrame
Columns: [Flow ID, Src IP, Src Port, Dst IP, Dst Port, Protocol, Timestamp, Flow Duration, Total Fwd Packet, Total Bwd packets, Total Length of Fwd Packet, Total Length of Bwd Packet, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std, Bwd Packet Length Max, Bwd Packet Length Min, Bwd Packet Length Mean, Bwd Packet Length Std, Flow Bytes/s, Flow Packets/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fw

In [ ]:
csecicids2018_column_map = {
    "Dst Port": "Dst Port",
    "Protocol": "Protocol",
    "Timestamp": "Timestamp",
    "Flow Duration": "Flow Duration",
    "Tot Fwd Pkts": "Total Fwd Packet",
    "Tot Bwd Pkts": "Total Bwd packets",
    "TotLen Fwd Pkts": "Total Length of Fwd Packet",
    "TotLen Bwd Pkts": "Total Length of Bwd Packet",
    "Fwd Pkt Len Max": "Fwd Packet Length Max",
    "Fwd Pkt Len Min": "Fwd Packet Length Min",
    "Fwd Pkt Len Mean": "Fwd Packet Length Mean",
    "Fwd Pkt Len Std": "Fwd Packet Length Std",
    "Bwd Pkt Len Max": "Bwd Packet Length Max",
    "Bwd Pkt Len Min": "Bwd Packet Length Min",
    "Bwd Pkt Len Mean": "Bwd Packet Length Mean",
    "Bwd Pkt Len Std": "Bwd Packet Length Std",
    "Flow Byts/s": "Flow Bytes/s",
    "Flow Pkts/s": "Flow Packets/s",
    "Flow IAT Mean": "Flow IAT Mean",
    "Flow IAT Std": "Flow IAT Std",
    "Flow IAT Max": "Flow IAT Max",
    "Flow IAT Min": "Flow IAT Min",
    "Fwd IAT Tot": "Fwd IAT Total",
    "Fwd IAT Mean": "Fwd IAT Mean",
    "Fwd IAT Std": "Fwd IAT Std",
    "Fwd IAT Max": "Fwd IAT Max",
    "Fwd IAT Min": "Fwd IAT Min",
    "Bwd IAT Tot": "Bwd IAT Total",
    "Bwd IAT Mean": "Bwd IAT Mean",
    "Bwd IAT Std": "Bwd IAT Std",
    "Bwd IAT Max": "Bwd IAT Max",
    "Bwd IAT Min": "Bwd IAT Min",
    "Fwd PSH Flags": "Fwd PSH Flags",
    "Bwd PSH Flags": "Bwd PSH Flags",
    "Fwd URG Flags": "Fwd URG Flags",
    "Bwd URG Flags": "Bwd URG Flags",
    "Fwd Header Len": "Fwd Header Length",
    "Bwd Header Len": "Bwd Header Length",
    "Fwd Pkts/s": "Fwd Packets/s",
    "Bwd Pkts/s": "Bwd Packets/s",
    "Pkt Len Min": "Packet Length Min",
    "Pkt Len Max": "Packet Length Max",
    "Pkt Len Mean": "Packet Length Mean",
    "Pkt Len Std": "Packet Length Std",
    "Pkt Len Var": "Packet Length Variance",
    "FIN Flag Cnt": "FIN Flag Count",
    "SYN Flag Cnt": "SYN Flag Count",
    "RST Flag Cnt": "RST Flag Count",
    "PSH Flag Cnt": "PSH Flag Count",
    "ACK Flag Cnt": "ACK Flag Count",
    "URG Flag Cnt": "URG Flag Count",
    "CWE Flag Count": "CWR Flag Count",
    "ECE Flag Cnt": "ECE Flag Count",
    "Down/Up Ratio": "Down/Up Ratio",
    "Pkt Size Avg": "Average Packet Size",
    "Fwd Seg Size Avg": "Fwd Segment Size Avg",
    "Bwd Seg Size Avg": "Bwd Segment Size Avg",
    "Fwd Byts/b Avg": "Fwd Bytes/Bulk Avg",
    "Fwd Pkts/b Avg": "Fwd Packet/Bulk Avg",
    "Fwd Blk Rate Avg": "Fwd Bulk Rate Avg",
    "Bwd Byts/b Avg": "Bwd Bytes/Bulk Avg",
    "Bwd Pkts/b Avg": "Bwd Packet/Bulk Avg",
    "Bwd Blk Rate Avg": "Bwd Bulk Rate Avg",
    "Subflow Fwd Pkts": "Subflow Fwd Packets",
    "Subflow Fwd Byts": "Subflow Fwd Bytes",
    "Subflow Bwd Pkts": "Subflow Bwd Packets",
    "Subflow Bwd Byts": "Subflow Bwd Bytes",
    "Init Fwd Win Byts": "FWD Init Win Bytes",
    "Init Bwd Win Byts": "Bwd Init Win Bytes",
    "Fwd Act Data Pkts": "Fwd Act Data Pkts",
    "Fwd Seg Size Min": "Fwd Seg Size Min",
    "Active Mean": "Active Mean",
    "Active Std": "Active Std",
    "Active Max": "Active Max",
    "Active Min": "Active Min",
    "Idle Mean": "Idle Mean",
    "Idle Std": "Idle Std",
    "Idle Max": "Idle Max",
    "Idle Min": "Idle Min",
    "Label": "Label"
}

# Call the ColumnStandardisation function for the CSE-CIC-IDS-2018 dataset
ColumnStandardisation(csecicids2018, csecicids2018_column_map)

csecicids2018_columns = pd.read_csv(f"{csecicids2018}/02-14-2018.csv", nrows=0)
print("\n" + csecicids2018_columns)

Finished processing 02-20-2018.csv
Finished processing 02-14-2018.csv
Finished processing 03-02-2018.csv
Finished processing 02-15-2018.csv
Finished processing 02-21-2018.csv
Finished processing 02-28-2018.csv
Finished processing 02-16-2018.csv
Finished processing 02-22-2018.csv
Finished processing 02-23-2018.csv
Finished processing 03-01-2018.csv
Empty DataFrame
Columns: [Dst Port, Protocol, Timestamp, Flow Duration, Total Fwd Packet, Total Bwd packets, Total Length of Fwd Packet, Total Length of Bwd Packet, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std, Bwd Packet Length Max, Bwd Packet Length Min, Bwd Packet Length Mean, Bwd Packet Length Std, Flow Bytes/s, Flow Packets/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Total, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Bwd IAT Total, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, Fwd Header Lengt

In [ ]:
cicddos2019_column_map = {
    "Flow ID": "Flow ID",
    "Source IP": "Src IP",
    "Source Port": "Src Port",
    "Destination IP": "Dst IP",
    "Destination Port": "Dst Port",
    "Protocol": "Protocol",
    "Timestamp": "Timestamp",
    "Flow Duration": "Flow Duration",
    "Total Fwd Packets": "Total Fwd Packet",
    "Total Backward Packets": "Total Bwd packets",
    "Total Length of Fwd Packets": "Total Length of Fwd Packet",
    "Total Length of Bwd Packets": "Total Length of Bwd Packet",
    "Fwd Packet Length Max": "Fwd Packet Length Max",
    "Fwd Packet Length Min": "Fwd Packet Length Min",
    "Fwd Packet Length Mean": "Fwd Packet Length Mean",
    "Fwd Packet Length Std": "Fwd Packet Length Std",
    "Bwd Packet Length Max": "Bwd Packet Length Max",
    "Bwd Packet Length Min": "Bwd Packet Length Min",
    "Bwd Packet Length Mean": "Bwd Packet Length Mean",
    "Bwd Packet Length Std": "Bwd Packet Length Std",
    "Flow Bytes/s": "Flow Bytes/s",
    "Flow Packets/s": "Flow Packets/s",
    "Flow IAT Mean": "Flow IAT Mean",
    "Flow IAT Std": "Flow IAT Std",
    "Flow IAT Max": "Flow IAT Max",
    "Flow IAT Min": "Flow IAT Min",
    "Fwd IAT Total": "Fwd IAT Total",
    "Fwd IAT Mean": "Fwd IAT Mean",
    "Fwd IAT Std": "Fwd IAT Std",
    "Fwd IAT Max": "Fwd IAT Max",
    "Fwd IAT Min": "Fwd IAT Min",
    "Bwd IAT Total": "Bwd IAT Total",
    "Bwd IAT Mean": "Bwd IAT Mean",
    "Bwd IAT Std": "Bwd IAT Std",
    "Bwd IAT Max": "Bwd IAT Max",
    "Bwd IAT Min": "Bwd IAT Min",
    "Fwd PSH Flags": "Fwd PSH Flags",
    "Bwd PSH Flags": "Bwd PSH Flags",
    "Fwd URG Flags": "Fwd URG Flags",
    "Bwd URG Flags": "Bwd URG Flags",
    "Fwd Header Length": "Fwd Header Length",
    "Bwd Header Length": "Bwd Header Length",
    "Fwd Packets/s": "Fwd Packets/s",
    "Bwd Packets/s": "Bwd Packets/s",
    "Min Packet Length": "Packet Length Min",
    "Max Packet Length": "Packet Length Max",
    "Packet Length Mean": "Packet Length Mean",
    "Packet Length Std": "Packet Length Std",
    "Packet Length Variance": "Packet Length Variance",
    "FIN Flag Count": "FIN Flag Count",
    "SYN Flag Count": "SYN Flag Count",
    "RST Flag Count": "RST Flag Count",
    "PSH Flag Count": "PSH Flag Count",
    "ACK Flag Count": "ACK Flag Count",
    "URG Flag Count": "URG Flag Count",
    "CWE Flag Count": "CWR Flag Count",
    "ECE Flag Count": "ECE Flag Count",
    "Down/Up Ratio": "Down/Up Ratio",
    "Average Packet Size": "Average Packet Size",
    "Avg Fwd Segment Size": "Fwd Segment Size Avg",
    "Avg Bwd Segment Size": "Bwd Segment Size Avg",
    "Fwd Avg Bytes/Bulk": "Fwd Bytes/Bulk Avg",
    "Fwd Avg Packets/Bulk": "Fwd Packet/Bulk Avg",
    "Fwd Avg Bulk Rate": "Fwd Bulk Rate Avg",
    "Bwd Avg Bytes/Bulk": "Bwd Bytes/Bulk Avg",
    "Bwd Avg Packets/Bulk": "Bwd Packet/Bulk Avg",
    "Bwd Avg Bulk Rate": "Bwd Bulk Rate Avg",
    "Subflow Fwd Packets": "Subflow Fwd Packets",
    "Subflow Fwd Bytes": "Subflow Fwd Bytes",
    "Subflow Bwd Packets": "Subflow Bwd Packets",
    "Subflow Bwd Bytes": "Subflow Bwd Bytes",
    "Init_Win_bytes_forward": "FWD Init Win Bytes",
    "Init_Win_bytes_backward": "Bwd Init Win Bytes",
    "act_data_pkt_fwd": "Fwd Act Data Pkts",
    "min_seg_size_forward": "Fwd Seg Size Min",
    "Active Mean": "Active Mean",
    "Active Std": "Active Std",
    "Active Max": "Active Max",
    "Active Min": "Active Min",
    "Idle Mean": "Idle Mean",
    "Idle Std": "Idle Std",
    "Idle Max": "Idle Max",
    "Idle Min": "Idle Min",
    "Label": "Label"
}

# Call the ColumnStandardisation function for the CIC-DDoS-2019 dataset
ColumnStandardisation(cicddos2019[0], cicddos2019_column_map)
ColumnStandardisation(cicddos2019[1], cicddos2019_column_map)

cicddos2019_columns = pd.read_csv(f"{cicddos2019[0]}/Portmap.csv", nrows=0)
print("\n" + cicddos2019_columns)

Finished processing LDAP.csv
Finished processing NetBIOS.csv
Finished processing MSSQL.csv
Finished processing UDP.csv
Finished processing Syn.csv
Finished processing UDPLag.csv
Finished processing Portmap.csv
Finished processing DrDoS_SSDP.csv
Finished processing DrDoS_NTP.csv
Finished processing DrDoS_LDAP.csv
Finished processing DrDoS_MSSQL.csv
Finished processing TFTP.csv
Finished processing DrDoS_DNS.csv
Finished processing DrDoS_NetBIOS.csv
Finished processing DrDoS_SNMP.csv
Finished processing DrDoS_UDP.csv
Finished processing Syn.csv
Finished processing UDPLag.csv
Empty DataFrame
Columns: [Unnamed: 0, Flow ID, Src IP, Src Port, Dst IP, Dst Port, Protocol, Timestamp, Flow Duration, Total Fwd Packet, Total Bwd packets, Total Length of Fwd Packet, Total Length of Bwd Packet, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std, Bwd Packet Length Max, Bwd Packet Length Min, Bwd Packet Length Mean, Bwd Packet Length Std, Flow Bytes/s, Flow Pack

In [ ]:
# Drop unnecessary columns
drop_columns = [
    "Flow ID", "Src IP", "Src Port", "Dst IP", "Timestamp",
    "Fwd Header Length.1", "SimillarHTTP", "Inbound", "Unnamed: 0"
]

# Iterate through all datasets
for dataset in datasets:

    # Iterate through all CSV files in each dataset folder
    for file in os.listdir(dataset):

        # Process only CSV files
        if file.endswith(".csv"):

            file_path = os.path.join(dataset, file)
            tmp_file_path = file_path + ".tmp"

            print(f"Processing file: {file}")

            # Track header writing for chunked output
            header_written = False

            # Read CSV file in chunks
            for chunk in pd.read_csv(file_path, chunksize=500_000, low_memory=False):

                # Drop only the columns that exist
                cols_to_drop = [c for c in drop_columns if c in chunk.columns]
                if cols_to_drop:
                    chunk.drop(columns=cols_to_drop, inplace=True)

                # Write chunk to temporary file
                chunk.to_csv(tmp_file_path, mode='a', index=False, header=not header_written)
                header_written = True

            # Replace original file with cleaned version
            os.replace(tmp_file_path, file_path)
            print(f"Finished {file}")

Processing file: CICFlowMeter_out.csv
Finished CICFlowMeter_out.csv
Processing file: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Finished Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Processing file: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Finished Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Processing file: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Finished Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Processing file: Tuesday-WorkingHours.pcap_ISCX.csv
Finished Tuesday-WorkingHours.pcap_ISCX.csv
Processing file: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Finished Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Processing file: Wednesday-workingHours.pcap_ISCX.csv
Finished Wednesday-workingHours.pcap_ISCX.csv
Processing file: Monday-WorkingHours.pcap_ISCX.csv
Finished Monday-WorkingHours.pcap_ISCX.csv
Processing file: Friday-WorkingHours-Morning.pcap_ISCX.csv
Finished Friday-WorkingHours-Morning.pcap_ISC

In [ ]:
def DuplicateHeaderCheck(directory, file):
    """ Checks CSV file for duplicate header rows appearing inside the data """
    
    file_path = os.path.join(directory, file)

    # Read file in chunks to handle large datasets efficiently
    chunk_iter = pd.read_csv(file_path, chunksize=500_000, low_memory=False)
    found_count = 0
    
    # Iterate through each chunk
    for chunk_idx, chunk in enumerate(chunk_iter):

        # Iterate through rows in the chunk
        for idx, row in chunk.iterrows():

            # Checks for value "Dst Port" in the Dst Port column
            if (row["Dst Port"]) == "Dst Port": 
                found_count += 1
                print(f"Header found at row {idx} in chunk {chunk_idx}")

    # Summary output
    if found_count == 0:
        print("No duplicate headers found.")
    else:
        print(f"Total duplicate headers found: {found_count}")

In [ ]:
# Iterate through all datasets and their files
for dataset in datasets:
    for file in os.listdir(dataset):

        print(f"\nScanning {file} for duplicate headers...")

        # Call the DuplicateHeaderCheck function to check for duplicate headers
        DuplicateHeaderCheck(dataset, file)


Scanning CICFlowMeter_out.csv for duplicate headers...
No duplicate headers found.

Scanning Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv for duplicate headers...
No duplicate headers found.

Scanning Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv for duplicate headers...
No duplicate headers found.

Scanning Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv for duplicate headers...
No duplicate headers found.

Scanning Tuesday-WorkingHours.pcap_ISCX.csv for duplicate headers...
No duplicate headers found.

Scanning Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv for duplicate headers...
No duplicate headers found.

Scanning Wednesday-workingHours.pcap_ISCX.csv for duplicate headers...
No duplicate headers found.

Scanning Monday-WorkingHours.pcap_ISCX.csv for duplicate headers...
No duplicate headers found.

Scanning Friday-WorkingHours-Morning.pcap_ISCX.csv for duplicate headers...
No duplicate headers found.

Scanning 02-20-2018.csv for duplicate headers.

1.3 - Data Merging

In [ ]:
# Output file where all datasets will be merged into a single CSV file
OUTPUT_FILE = "data/original/merged_data.csv"

# Track whether CSV header has already been written
header_written = False

# Iterate through each dataset
for dataset in datasets:

    # Extract dataset name
    dataset_name = dataset.split("/")[2]
    print("Processing dataset:", dataset_name)

    # Iterate through all files in the dataset (sorted to preserve correct temporal ordering)
    for file in sorted(os.listdir(dataset)):

        file_path = os.path.join(dataset, file)
        print("Merging file:", file)

        # Read CSV file in chunks
        for chunk in pd.read_csv(file_path, chunksize=500_000, low_memory=False):

            # Add dataset source label to each row
            chunk["Dataset"] = dataset_name
            
            # Append chunk to final merged CSV file
            chunk.to_csv(OUTPUT_FILE, mode="a", index=False, header=not header_written)

            # Ensure header is only written once
            header_written = True

print("\nAll datasets merged successfully: merged_data.csv")

Processing dataset: cic-unsw-nb15
Merging file: CICFlowMeter_out.csv
Processing dataset: cic-ids-2017
Merging file: 01-Monday-WorkingHours.pcap_ISCX.csv
Merging file: 02-Tuesday-WorkingHours.pcap_ISCX.csv
Merging file: 03-Wednesday-workingHours.pcap_ISCX.csv
Merging file: 04-Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Merging file: 05-Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Merging file: 06-Friday-WorkingHours-Morning.pcap_ISCX.csv
Merging file: 07-Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Merging file: 08-Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Processing dataset: cse-cic-ids-2018
Merging file: 02-14-2018.csv
Merging file: 02-15-2018.csv
Merging file: 02-16-2018.csv
Merging file: 02-20-2018.csv
Merging file: 02-21-2018.csv
Merging file: 02-22-2018.csv
Merging file: 02-23-2018.csv
Merging file: 02-28-2018.csv
Merging file: 03-01-2018.csv
Merging file: 03-02-2018.csv
Processing dataset: cic-ddos-2019
Merging file: 01-Portmap.csv
Merging

In [ ]:
print(f"\nScanning {OUTPUT_FILE} for duplicate headers...")

# Call the DuplicateHeaderCheck function to check for duplicate headers
DuplicateHeaderCheck("data/original/", "merged_data.csv")


Scanning data/original/merged_data.csv for duplicate headers...
No duplicate headers found.


1.4 - Data Cleaning

In [ ]:
INPUT_FILE = "data/original/merged_data.csv"
OUTPUT_FILE = "data/cleaned/merged_data_cleaned.csv"

# Track whether header has been written to output file
header_written = False

# Read merged dataset in chunks
for chunk in pd.read_csv(INPUT_FILE, chunksize=500_000, low_memory=False):

    # Replace all infinite values to NaN
    chunk = chunk.replace([np.inf, -np.inf], np.nan) 

    # Removes rows with NaN values
    chunk = chunk.dropna() 

    # Append cleaned chunk to output file
    chunk.to_csv(
        OUTPUT_FILE,
        mode="a",
        index=False,
        header=not header_written
    )
    
    header_written = True

print("Initial data cleaning complete")

Initial data cleaning complete


In [ ]:
# Check Dtype of columns
data = pd.read_csv("data/cleaned/merged_data_cleaned.csv", nrows=5)
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 81 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Dst Port                    5 non-null      int64  
 1   Protocol                    5 non-null      int64  
 2   Timestamp                   5 non-null      str    
 3   Flow Duration               5 non-null      int64  
 4   Total Fwd Packet            5 non-null      int64  
 5   Total Bwd packets           5 non-null      int64  
 6   Total Length of Fwd Packet  5 non-null      float64
 7   Total Length of Bwd Packet  5 non-null      float64
 8   Fwd Packet Length Max       5 non-null      float64
 9   Fwd Packet Length Min       5 non-null      float64
 10  Fwd Packet Length Mean      5 non-null      float64
 11  Fwd Packet Length Std       5 non-null      float64
 12  Bwd Packet Length Max       5 non-null      float64
 13  Bwd Packet Length Min       5 non-null      float6

In [ ]:
INPUT_FILE = "data/cleaned/merged_data_cleaned.csv"
TMP_FILE = "data/cleaned/merged_data_final.tmp"
OUTPUT_FILE = INPUT_FILE

# Columns treated as categorical
CATEGORICAL_COLS = [
    "Protocol", "Label", "Dataset", "Dst Port", "Timestamp"
]

# Columns explicitly cast to int8
INT8_COLS = [
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count",
    "PSH Flag Count", "ACK Flag Count", "URG Flag Count",
    "CWR Flag Count", "ECE Flag Count",
    "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags"
]

# Columns explicitly cast to int32
INT32_COLS = [
    "Total Fwd Packet", "Total Bwd packets",
    "Fwd Header Length", "Bwd Header Length",
    "Subflow Fwd Packets", "Subflow Bwd Packets",
    "Subflow Fwd Bytes", "Subflow Bwd Bytes",
    "Fwd Act Data Pkts", "Fwd Seg Size Min",
    "FWD Init Win Bytes", "Bwd Init Win Bytes"
]

# Columns cast to float32
FLOAT32_COLS = [
    "Flow Duration", "Total Length of Fwd Packet",
    "Total Length of Bwd Packet", "Fwd Packet Length Max",
    "Fwd Packet Length Min", "Fwd Packet Length Mean",
    "Fwd Packet Length Std", "Bwd Packet Length Max",
    "Bwd Packet Length Min", "Bwd Packet Length Mean",
    "Bwd Packet Length Std", "Flow Bytes/s", "Flow Packets/s",
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max",
    "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std",
    "Bwd IAT Max", "Bwd IAT Min", "Fwd Packets/s", "Bwd Packets/s",
    "Packet Length Min", "Packet Length Max", "Down/Up Ratio",
    "Packet Length Mean", "Packet Length Std",
    "Packet Length Variance", "Average Packet Size",
    "Fwd Segment Size Avg", "Bwd Segment Size Avg",
    "Fwd Bytes/Bulk Avg", "Fwd Packet/Bulk Avg", "Fwd Bulk Rate Avg",
    "Bwd Bytes/Bulk Avg", "Bwd Packet/Bulk Avg", "Bwd Bulk Rate Avg",
    "Active Mean", "Active Std", "Active Max", "Active Min",
    "Idle Mean", "Idle Std", "Idle Max", "Idle Min"
]

# Build dtype mapping for memory optimisation
DTYPE_MAP = {col: "float32" for col in FLOAT32_COLS}
DTYPE_MAP.update({col: "int32" for col in INT32_COLS})
DTYPE_MAP.update({col: "int8" for col in INT8_COLS})

# Track header writing for chunked output
header_written = False

# Track globally seen rows for deduplication
seen_hashes = set()

# Process dataset in chunks
for chunk in pd.read_csv(INPUT_FILE, chunksize=500_000, low_memory=False):

    # Local deduplication (CHUNK)
    chunk = chunk.drop_duplicates()

    # Convert columns safely to numeric
    for col in DTYPE_MAP:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    # Replace infinities
    chunk.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Cast numeric dtypes
    for col, dtype in DTYPE_MAP.items():
        if col in chunk.columns:
            chunk[col] = chunk[col].astype(dtype)

    # Cast categoricals
    for col in CATEGORICAL_COLS:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("category")

    # Global deduplication (through hashing)
    row_hashes = pd.util.hash_pandas_object(chunk, index=False)

    # Keep only unseen rows
    mask = ~row_hashes.isin(seen_hashes)
    chunk = chunk.loc[mask]

    # Update global hash set
    seen_hashes.update(row_hashes[mask])

    # Write to temporary file
    chunk.to_csv(
        TMP_FILE,
        mode="a",
        index=False,
        header=not header_written
    )
    header_written = True

# Replace old file with final processed version
os.replace(TMP_FILE, OUTPUT_FILE)

print(f"Dtype standardisation + deduplication complete: {OUTPUT_FILE}")

Dtype standardisation + deduplication complete: data/cleaned/merged_data_cleaned.csv


In [ ]:
print(f"\nScanning {OUTPUT_FILE} for duplicate headers...")

# Call the DuplicateHeaderCheck function to check for duplicate headers
DuplicateHeaderCheck("data/cleaned/", "merged_data_cleaned.csv")


Scanning data/cleaned/merged_data_cleaned.csv for duplicate headers...
No duplicate headers found.


In [ ]:
# Store total label counts across all chunks
TotalCount = pd.Series(dtype="int") 

# Read only the label column in chunks
for chunk in pd.read_csv("data/cleaned/merged_data_cleaned.csv", usecols=["Label"], low_memory=False, chunksize=500_000):
    
    # Count label occurrences in current chunk
    valueCount = chunk["Label"].value_counts()

    # Accumulate counts across all chunks
    TotalCount = TotalCount.add(valueCount, fill_value=0)
        
# Display final label distribution
print("\nTotal Label Counts:")
print(TotalCount.astype(int).sort_values(ascending=False))


Total Label Counts:
Label
Benign                        12910940
TFTP                          12305614
MSSQL                          5455244
DrDoS_MSSQL                    4312355
UDP                            3214828
DrDoS_SNMP                     3161477
Syn                            3092122
DrDoS_DNS                      2717089
DrDoS_UDP                      2656705
BENIGN                         2270821
DrDoS_SSDP                     2267039
DrDoS_NetBIOS                  1747685
NetBIOS                        1717410
DrDoS_LDAP                     1364653
DrDoS_NTP                      1194140
LDAP                           1131424
DDoS attacks-LOIC-HTTP          575715
UDP-lag                         330079
DDOS attack-HOIC                308806
DoS attacks-Hulk                281217
DoS Hulk                        178179
Portmap                         177197
Bot                             166223
Infilteration                   155094
DDoS                            12802